### Prepare Data for Player Embeddings

In [47]:
import sys
from pathlib import Path

# Regarding Tennis_ML_Project/Notebooks, scr is a sibbling, not a child, 
# so we need to add the parent of Notebooks to the path so that we can import from src

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))


In [48]:
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.dataset import build_player_id_mapping, add_player_ids, TennisMatchDatasetWithPlayers


In [49]:
matches = pd.read_csv("../Data/processed/matches_with_players_clusters.csv")
matches.drop(columns=["winner_id", "loser_id"], inplace=True)  # Remove old player ID columns if they exist
matches.head()


,year,tourney_date,surface,tourney_level,winner_name,winner_rank,winner_age,winner_ht,loser_name,loser_rank,...,w_ace,w_df,w_bpSaved,w_bpFaced,l_ace,l_df,l_bpSaved,l_bpFaced,winner_cluster,loser_cluster
0,2018,20180101,Hard,A,Ryan Harrison,47.0,25.6,185.0,Leonardo Mayer,52.0,...,9.0,2.0,8.0,9.0,10.0,3.0,1.0,4.0,4,4
1,2018,20180101,Hard,A,Jared Donaldson,54.0,21.2,188.0,Jordan Thompson,94.0,...,5.0,3.0,4.0,5.0,3.0,5.0,7.0,11.0,4,3
2,2018,20180101,Hard,A,Denis Istomin,63.0,31.3,185.0,Damir Dzumhur,30.0,...,7.0,0.0,9.0,11.0,8.0,6.0,10.0,16.0,1,3
3,2018,20180101,Hard,A,Alex De Minaur,208.0,18.8,183.0,Steve Johnson,44.0,...,9.0,3.0,2.0,3.0,6.0,2.0,4.0,6.0,0,4
4,2018,20180101,Hard,A,Michael Mmoh,175.0,19.9,188.0,Federico Delbonis,68.0,...,5.0,4.0,3.0,3.0,4.0,0.0,0.0,2.0,4,1


#### Build Player Mapping

In [50]:
player_to_id, id_to_player = build_player_id_mapping(matches)

num_players = len(player_to_id)
print(f"Number of players with embeddings: {num_players}")


Number of players with embeddings: 924


In [51]:
matches = add_player_ids(matches, player_to_id)

matches[["winner_name", "winner_id", "loser_name", "loser_id"]].head()
# matches_a = matches[matches["winner_name"] == "Alex De Minaur"]
# matches_a[["winner_name", "winner_id", "loser_name", "loser_id"]].head()


,winner_name,winner_id,loser_name,loser_id
0,Ryan Harrison,0,Leonardo Mayer,55
1,Jared Donaldson,1,Jordan Thompson,258
2,Denis Istomin,2,Damir Dzumhur,56
3,Alex De Minaur,3,Steve Johnson,51
4,Michael Mmoh,4,Federico Delbonis,176


In [52]:
matches[["winner_name", "winner_id", "loser_name", "loser_id"]].isna().sum()

winner_name    0
winner_id      0
loser_name     0
loser_id       0
dtype: int64

In [53]:
matches.columns

Index(['year', 'tourney_date', 'surface', 'tourney_level', 'winner_name',
       'winner_rank', 'winner_age', 'winner_ht', 'loser_name', 'loser_rank',
       'loser_age', 'loser_ht', 'w_ace', 'w_df', 'w_bpSaved', 'w_bpFaced',
       'l_ace', 'l_df', 'l_bpSaved', 'l_bpFaced', 'winner_cluster',
       'loser_cluster', 'winner_id', 'loser_id'],
      dtype='object')

At this point, every player has a stable ID, mapping is reusable, no leakage, ready for embeddings

#### Build Match-Level Features

In [54]:
import numpy as np

def build_match_dataset(matches):
    rows = []

    for _, row in matches.iterrows():
        # winner vs loser
        rows.append({
            "winner_id": row["winner_id"],
            "loser_id": row["loser_id"],
            "rank_diff": row["winner_rank"] - row["loser_rank"],
            "age_diff": row["winner_age"] - row["loser_age"],
            "height_diff": row["winner_ht"] - row["loser_ht"],
            "cluster_diff": row["winner_cluster"] - row["loser_cluster"],
            "surface": row["surface"],
            "tourney_level": row["tourney_level"],
            "target": 1
        })
        
        # loser vs winner
        rows.append({
            "winner_id": row["loser_id"],
            "loser_id": row["winner_id"],
            "rank_diff": row["loser_rank"] - row["winner_rank"],
            "age_diff": row["loser_age"] - row["winner_age"],
            "height_diff": row["loser_ht"] - row["winner_ht"],
            "cluster_diff": row["loser_cluster"] - row["winner_cluster"],
            "surface": row["surface"],
            "tourney_level": row["tourney_level"],
            "target": 0
        })
    
    return pd.DataFrame(rows)

dataset = build_match_dataset(matches)

print(dataset.shape)
dataset.head()


(37754, 9)


,winner_id,loser_id,rank_diff,age_diff,height_diff,cluster_diff,surface,tourney_level,target
0,0,55,-5.0,-5.0,-3.0,0,Hard,A,1
1,55,0,5.0,5.0,3.0,0,Hard,A,0
2,1,258,-40.0,-2.5,5.0,1,Hard,A,1
3,258,1,40.0,2.5,-5.0,-1,Hard,A,0
4,2,56,33.0,5.7,10.0,-2,Hard,A,1


### Extend the PyTorch Dataset to Support Player IDs

#### Create Datasets

#### Time Split

In [55]:
train_matches = matches[matches["year"] <= 2022]
val_matches = matches[matches["year"] == 2023]
test_matches = matches[matches["year"] == 2024]

train_data = build_match_dataset(train_matches)
val_data = build_match_dataset(val_matches)
test_data = build_match_dataset(test_matches)

print(train_data.shape, val_data.shape, test_data.shape)

(25630, 9) (5972, 9) (6152, 9)


#### Handle Missing Values

In [56]:
from sklearn.impute import SimpleImputer

# Even though we don't have missing values in cluster_diff, we include it here for consistency
# The imputer does nothing to this column
numeric_features = ["rank_diff", "age_diff", "height_diff", "cluster_diff"]
categorical_features = ["surface", "tourney_level"]
target_col = "target"

num_imputer = SimpleImputer(strategy="median")

train_data[numeric_features] = num_imputer.fit_transform(train_data[numeric_features])
val_data[numeric_features] = num_imputer.transform(val_data[numeric_features])
test_data[numeric_features] = num_imputer.transform(test_data[numeric_features])


In [57]:
cat_imputer = SimpleImputer(strategy="most_frequent")

train_data[["surface"]] = cat_imputer.fit_transform(train_data[["surface"]])
val_data[["surface"]] = cat_imputer.transform(val_data[["surface"]])
test_data[["surface"]] = cat_imputer.transform(test_data[["surface"]])


In [58]:
print(train_data.isna().sum())
print(val_data.isna().sum())
print(test_data.isna().sum())


winner_id        0
loser_id         0
rank_diff        0
age_diff         0
height_diff      0
cluster_diff     0
surface          0
tourney_level    0
target           0
dtype: int64
winner_id        0
loser_id         0
rank_diff        0
age_diff         0
height_diff      0
cluster_diff     0
surface          0
tourney_level    0
target           0
dtype: int64
winner_id        0
loser_id         0
rank_diff        0
age_diff         0
height_diff      0
cluster_diff     0
surface          0
tourney_level    0
target           0
dtype: int64


#### Extract X / y from Splits

In [59]:
X_train = train_data[numeric_features + categorical_features]
X_val   = val_data[numeric_features + categorical_features]
X_test  = test_data[numeric_features + categorical_features]

y_train = train_data[target_col].values
y_val   = val_data[target_col].values
y_test  = test_data[target_col].values



#### Encode Categoricals (OneHot)

In [60]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_cat = encoder.fit_transform(X_train[categorical_features])
X_val_cat   = encoder.transform(X_val[categorical_features])
X_test_cat  = encoder.transform(X_test[categorical_features])


#### Scale Numeric Features

In [61]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train[numeric_features])
X_val_num   = scaler.transform(X_val[numeric_features])
X_test_num  = scaler.transform(X_test[numeric_features])


In [62]:
print(X_train_num.shape, X_train_cat.shape)
print(X_val_num.shape, X_val_cat.shape)
print(X_test_num.shape, X_test_cat.shape)


(25630, 4) (25630, 8)
(5972, 4) (5972, 8)
(6152, 4) (6152, 8)


#### Extract Player IDs

In [63]:
winner_ids_train = train_data["winner_id"].values
loser_ids_train  = train_data["loser_id"].values

winner_ids_val = val_data["winner_id"].values
loser_ids_val  = val_data["loser_id"].values

winner_ids_test = test_data["winner_id"].values
loser_ids_test  = test_data["loser_id"].values


#### Create Datasets and DataLoaders

In [64]:
train_dataset = TennisMatchDatasetWithPlayers(
    X_train_num,
    X_train_cat,
    winner_ids_train,
    loser_ids_train,
    y_train
)

print(type(train_dataset.y))


val_dataset = TennisMatchDatasetWithPlayers(
    X_val_num,
    X_val_cat,
    winner_ids_val,
    loser_ids_val,
    y_val
)

print(type(val_dataset.y))

test_dataset = TennisMatchDatasetWithPlayers(
    X_test_num,
    X_test_cat,
    winner_ids_test,
    loser_ids_test,
    y_test
)


<class 'torch.Tensor'>
<class 'torch.Tensor'>


In [65]:
type(y_train)


numpy.ndarray

In [66]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(train_loader)


In [67]:
batch = next(iter(train_loader))

for tensor in batch:
    print(tensor.shape, tensor.dtype)


torch.Size([64, 4]) torch.float32
torch.Size([64, 8]) torch.float32
torch.Size([64]) torch.int64
torch.Size([64]) torch.int64
torch.Size([64]) torch.float32
